# B3 · GeoDataFrames and file formats: Nepal's districts as a table

> **Time:** about 90 minutes  
> **You need:** Lessons B1 and B2, and all of Part A, especially paths (A7) and pandas tables (A9).  
> **By the end you can:**
> 1. read a vector file into a **GeoDataFrame** and inspect it: rows, columns, CRS, bounds and geometry types,
> 2. turn a CSV file with longitude and latitude columns into a point layer, and map polygons and points together,
> 3. save layers as GeoPackage, Shapefile and GeoJSON files, explain the strengths and weak points of each, and read them back.

In Lesson B2 you built shapes one at a time. Real GIS data comes in files that hold hundreds or millions of shapes, each with attributes. **GeoPandas** reads such a file into a table you already know how to use: a pandas DataFrame, with one extra column that holds the shapes.

In this lesson you load the 77 districts of Nepal and turn the peak and landslide CSV files into maps. Then you save your layers in the three file formats you will meet most often.

**Counting lines:** `fig, ax = plt.subplots()` and `plt.show()` do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 13 of 21 · [Course home](../README.md)  
← Previous: [B2 · Points, lines and polygons: geometry with shapely](B02_Points_lines_and_polygons.ipynb) · Next: [B4 · Coordinate reference systems in practice: measuring Nepal in metres](B04_Coordinate_reference_systems.ipynb) →

## 1. From GeoSeries to GeoDataFrame

In Lesson B2 you put geometries in a **GeoSeries**. A **GeoDataFrame** is a pandas DataFrame with one special column that holds a GeoSeries: the **geometry column**.

Each row is one feature (Lesson B1): its attributes sit in ordinary columns, and its shape sits in the geometry column.

Everything from Lesson A9 still works: selecting columns, filtering rows, sorting and grouping. On top of that, a GeoDataFrame knows spatial things, such as its coordinate reference system (`.crs`), its bounding box (`.total_bounds`) and how to draw itself (`.plot()`).

To build one, give a table, a list of geometries and the CRS: `gpd.GeoDataFrame(table, geometry=shapes, crs="EPSG:4326")`. EPSG:4326 means "longitude and latitude in degrees"; Lesson B4 explains it.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely import Point

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

stops = pd.DataFrame({
    "place": ["Lukla", "Namche Bazaar", "Everest Base Camp"],
    "elevation_m": [2860, 3440, 5364],
})
shapes = [Point(86.7299, 27.6866), Point(86.7139, 27.8047), Point(86.8528, 28.0029)]

stops_gdf = gpd.GeoDataFrame(stops, geometry=shapes, crs="EPSG:4326")
stops_gdf

In [ ]:
print(type(stops_gdf))
print(type(stops_gdf.geometry))
print(stops_gdf.crs)

The table looks like a DataFrame with a `geometry` column that holds one `POINT` per row. `stops_gdf.geometry` is the GeoSeries inside it, and `.crs` reports `EPSG:4326`.

### Practice

**Q1 · one line:** Print the mean elevation of the three stops in `stops_gdf`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(stops_gdf["elevation_m"].mean())
```

About 3,888 m. Attribute columns behave exactly as in pandas.
</details>

**Q2 · two lines:** Print the x values, then the y values, of all points in `stops_gdf`, using `.geometry.x` and `.geometry.y`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(stops_gdf.geometry.x)
print(stops_gdf.geometry.y)
```

Each is a Series with one number per row: the longitudes (86.7299, 86.7139, 86.8528), then the latitudes. In Lesson B2 `.x` worked on one point; on a GeoSeries it works on every point at once.
</details>

**Q3 · three lines:** Build your own GeoDataFrame `two_peaks` with a `name` and a `height_m` column for Everest (86.925, 27.9881; 8,849 m) and Lhotse (86.9333, 27.9617; 8,516 m), in EPSG:4326. Show it.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
table = pd.DataFrame({"name": ["Everest", "Lhotse"], "height_m": [8849, 8516]})
two_peaks = gpd.GeoDataFrame(table, geometry=[Point(86.925, 27.9881), Point(86.9333, 27.9617)], crs="EPSG:4326")
two_peaks
```

A table with two rows, two attributes and a geometry column. The list of points must be in the same order as the rows.
</details>

## 2. Reading a vector file

`gpd.read_file(path)` reads a vector file, such as GeoJSON, GeoPackage or Shapefile, into a GeoDataFrame. One line, whatever the format.

Then look before you analyse, as in Lesson A9. On top of `head()`, `shape` and `columns`, check three spatial things:

| Code | Shows |
| --- | --- |
| `gdf.crs` | the coordinate reference system |
| `gdf.geom_type` | the kind of shape in each row |
| `gdf.total_bounds` | `[min_x, min_y, max_x, max_y]` of the whole layer |

A **MultiPolygon** is one feature made of one or more polygon parts, for example a district with an island. Files often store every area as a MultiPolygon, even when it has only one part, so that all rows share the same type.

**Predict:** how many rows will the district file have?

In [ ]:
districts = gpd.read_file(DATA / "nepal_districts.geojson")
districts.head()

In [ ]:
print(districts.shape)
print(districts.crs)
print(districts.geom_type.value_counts())
print(districts.total_bounds)

The file holds 77 rows, one per district, and 4 columns: `district`, `district_code`, `province_code` and `geometry`. All 77 shapes are MultiPolygons, the CRS is EPSG:4326, and the layer spans 80.06° to 88.20° E and 26.35° to 30.45° N, as in Lesson B1.

District names follow the spelling of the source data, for example **Chitawan** and **Kabhrepalanchok**. Always copy names from the data, not from memory.

### Practice

**Q1 · one line:** Print the column names of `districts`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(districts.columns)
```

`district`, `district_code`, `province_code` and `geometry`. The province *names* are in a separate table, `province_names.csv`, which Lesson B6 joins on.
</details>

**Q2 · two lines:** Read `nepal_provinces.geojson` into `provinces`. Print its shape and its geometry types.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
print(provinces.shape, provinces.geom_type.value_counts())
```

7 rows and 2 columns (`province_code` and `geometry`), all of type Polygon. Each province is one piece of land, so one polygon is enough.
</details>

**Q3 · three lines:** Read `district_headquarters.geojson` into `hq`. Print the number of rows and the geometry types, then show the first 3 rows.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq = gpd.read_file(DATA / "district_headquarters.geojson")
print(len(hq), hq.geom_type.value_counts())
hq.head(3)
```

75 points, each with a `name`. Why 75 and not 77? This layer was made before Nawalparasi and Rukum were each split in two. Always check how up to date a dataset is.
</details>

## 3. A GeoDataFrame is still a pandas table

Every tool from Lesson A9 works on a GeoDataFrame:

- `districts["district"]` selects a column,
- `districts[districts["province_code"] == "NP03"]` filters rows with a mask,
- `value_counts()`, `sort_values()` and `groupby()` work as before.

The geometry column travels with its row. When you filter, each district keeps its own shape, so the result is still a GeoDataFrame that can be drawn.

The province codes run from NP01 to NP07. Lesson B6 matches them to the names Koshi, Madhesh, Bagmati, Gandaki, Lumbini, Karnali and Sudurpashchim.

In [ ]:
print(districts["province_code"].value_counts().sort_index())

In [ ]:
kathmandu = districts[districts["district"] == "Kathmandu"]
print(type(kathmandu))
kathmandu

In [ ]:
kathmandu.plot()

NP01 has the most districts (14) and NP02 the fewest (8). Filtering gave a one-row GeoDataFrame, which `.plot()` draws as Kathmandu district's outline. `sort_index()` sorted the counts by their labels, NP01 to NP07.

### Practice

**Q1 · one line:** How many districts are in province NP03? Count the `True` values of a mask.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(districts["province_code"] == "NP03").sum()
```

13 districts. As in Lesson A9, `.sum()` on a mask counts the rows where it is `True`.
</details>

**Q2 · two lines:** Find the row for Chitwan district (spelled `Chitawan` in this file) and print its `district_code`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
chitawan = districts[districts["district"] == "Chitawan"]
print(chitawan["district_code"].iloc[0])
```

It prints `NP0335`. Searching for "Chitwan" would give an empty table, a mismatch you will meet again in Lesson B6.
</details>

**Q3 · three lines:** Select the districts of province NP06 into `karnali`. Print their names, sorted alphabetically, and plot them.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
karnali = districts[districts["province_code"] == "NP06"]
print(karnali["district"].sort_values().tolist())
karnali.plot()
```

Ten districts, from Dailekh to Surkhet, including Rukum West. `.tolist()` turns the Series into a plain list, which prints on one line. The map shows a whole province, built from its districts.
</details>

## 4. Mapping a GeoDataFrame

`.plot()` draws every geometry in the layer. A few options make the map clear:

- `column="province_code"` colours each feature by an attribute. Add `categorical=True` for text categories, and `legend=True` for a legend.
- `cmap="Set2"` picks a set of colours (a **colour map**). Set2 has soft, clearly different colours, good for categories that have no order.
- `edgecolor` and `linewidth` draw the borders.
- `facecolor="none"` draws outlines only, and `color="lightgrey"` gives one plain fill.

To draw several layers on one map, pass the same axes: `layer.plot(ax=ax)`. The layer drawn last is on top. Lesson B11 turns these maps into finished, publishable ones.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
districts.plot(ax=ax, column="province_code", categorical=True, legend=True,
               cmap="Set2", edgecolor="white", linewidth=0.5)
ax.set_title("The 77 districts of Nepal, coloured by province code")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
districts.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5)
kathmandu.plot(ax=ax, color="tab:red")
ax.set_title("Kathmandu district")
plt.show()

GeoPandas set the aspect for you, and labelled the axes from the CRS ("Geodetic longitude [degree]"); the title is yours. The legend lists the seven province codes; Lesson B6 replaces them with names. In the second map, Kathmandu district is small compared with the mountain districts of the north and west.

### Practice

**Q1 · one line:** Draw the district outlines only, in black, with no fill.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts.plot(facecolor="none", edgecolor="black")
```

With `facecolor="none"` only the borders are drawn: the same result as `districts.boundary.plot()` in Lesson B1.
</details>

**Q2 · two lines:** Read `nepal_provinces.geojson` and draw the provinces coloured by `province_code`, with a legend.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
provinces.plot(column="province_code", categorical=True, legend=True, cmap="Set2")
```

Seven areas, one per province. They match the colours of the district map, because the provinces were made by merging the districts (see `data/README.md`).
</details>

**Q3 · three lines:** Draw all districts in light grey, with the ten Karnali (NP06) districts in red on top. Add a title.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
districts[districts["province_code"] == "NP06"].plot(ax=ax, color="tab:red", edgecolor="white")
ax.set_title("Districts of Karnali (NP06)")
plt.show()
```

Karnali covers a large part of the north-west. The grey layer gives context, and the red layer answers the question "where?".
</details>

## 5. Points from a CSV file

Many datasets arrive as a CSV file with longitude and latitude columns: GPS logs, field records, and the practice landslides.

`gpd.points_from_xy(x, y)` turns two columns into points in one line. As always, **x is longitude**: `gpd.points_from_xy(df["lon"], df["lat"])`. It does the loop you wrote in Lesson B2 for you.

A CSV file does not say which CRS its numbers use. You must know it and state it with `crs="EPSG:4326"`. GPS coordinates in decimal degrees are almost always EPSG:4326.

The `lon` and `lat` columns stay in the table as ordinary attributes.

In [ ]:
slides = pd.read_csv(DATA / "practice_landslides.csv")
slides_gdf = gpd.GeoDataFrame(
    slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326"
)

peaks = pd.read_csv(DATA / "nepal_peaks.csv")
peaks_gdf = gpd.GeoDataFrame(
    peaks, geometry=gpd.points_from_xy(peaks["lon"], peaks["lat"]), crs="EPSG:4326"
)
slides_gdf.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
districts.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5)
slides_gdf.plot(ax=ax, color="tab:orange", markersize=15, label="Landslides (practice data)")
peaks_gdf.plot(ax=ax, color="tab:red", marker="^", markersize=50, label="Peaks above 8,000 m")
ax.set_title("Practice landslides and the eight peaks")
ax.legend(loc="lower left")
plt.show()

Each landslide record now has a `POINT` geometry. On the map, the landslides form clusters in a few districts, and the peaks line the northern border. `label=` gives each layer a legend entry, as in Lesson A10.

### Practice

**Q1 · one line:** Show the geometry column of the first 3 rows of `peaks_gdf`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks_gdf.geometry.head(3)
```

`POINT (86.925 27.9881)` for Everest, then Kanchenjunga and Lhotse, in the same order as the CSV rows. The dtype at the bottom says `geometry`.
</details>

**Q2 · two lines:** Print how many landslides there are per `trigger` in `slides_gdf`, then the layer's `total_bounds`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(slides_gdf["trigger"].value_counts())
print(slides_gdf.total_bounds)
```

29 rainfall, 8 road cut and 3 earthquake records, as in Lesson A9. The points span about 83.42° to 87.86° E and 27.40° to 28.64° N, a much smaller box than the whole country.
</details>

**Q3 · three lines:** Turn `everest_trek.csv` into a point layer `trek_gdf`, then draw Solukhumbu district in light grey with the trek stops on top.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
trek = pd.read_csv(DATA / "everest_trek.csv")
trek_gdf = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]), crs="EPSG:4326")
ax = districts[districts["district"] == "Solukhumbu"].plot(color="lightgrey")
trek_gdf.plot(ax=ax, color="tab:blue")
plt.show()
```

All eight stops fall inside Solukhumbu district. `.plot()` returns the axes it drew on, so `ax = ....plot()` lets the next layer use the same axes without `plt.subplots()`.
</details>

## 6. Saving layers: GeoPackage, Shapefile and GeoJSON

`gdf.to_file(path)` writes a layer to a file. The file ending picks the format. You will meet three formats again and again:

| Format | Files on disk | Strong points | Weak points |
| --- | --- | --- | --- |
| **GeoJSON** (`.geojson`) | one text file | human-readable; used on the web | large and slow for big data; normally longitude and latitude only |
| **Shapefile** (`.shp`) | several files that must stay together | read by every GIS program | column names of at most 10 characters; one layer per file; at most 2 GB |
| **GeoPackage** (`.gpkg`) | one file (a small database) | compact; many layers in one file; long column names | not human-readable |

For your own work, GeoPackage is a good default.

`path.stat().st_size` gives a file's size in bytes, using `pathlib` from Lesson A7.

In [ ]:
districts.to_file(OUT / "b3_districts.geojson")
districts.to_file(OUT / "b3_districts.shp")
districts.to_file(OUT / "b3_nepal.gpkg", layer="districts")

In [ ]:
for path in sorted(OUT.glob("b3_*")):
    print(f"{path.name:22} {path.stat().st_size:>9,} bytes")

Writing the Shapefile printed **warnings** (pink boxes), not errors. They say that `district_code` and `province_code` are longer than 10 characters, so they were cut to `district_c` and `province_c`. Step 7 shows the result.

One Shapefile is really five files with the same name:

| File | Holds |
| --- | --- |
| `.shp` | the geometries |
| `.shx` | an index that helps find each geometry in `.shp` |
| `.dbf` | the attribute table |
| `.prj` | the CRS, as text |
| `.cpg` | the text encoding of the attributes |

Copy or email all of them together; a `.shp` without its `.dbf` and `.prj` is close to useless. Here the GeoJSON is the largest file (747,241 bytes), the GeoPackage is smaller (466,944 bytes), and the Shapefile's five parts together are smaller still.

### Practice

**Q1 · one line:** Print the text inside `b3_districts.prj`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((OUT / "b3_districts.prj").read_text())
```

It starts `GEOGCS["GCS_WGS_1984"`: a text description of the CRS, longitude and latitude on the WGS 84 datum. Without this file, a program would not know what the numbers in `.shp` mean.
</details>

**Q2 · two lines:** How many times larger is the GeoJSON file than the GeoPackage file? Store both sizes, then print the ratio rounded to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
geojson_bytes = (OUT / "b3_districts.geojson").stat().st_size
print(round(geojson_bytes / (OUT / "b3_nepal.gpkg").stat().st_size, 1))
```

About 1.6 times. GeoJSON writes every coordinate as text, digit by digit, while GeoPackage stores numbers in a compact binary form.
</details>

**Q3 · three lines:** Save `slides_gdf` as `b3_landslides.geojson` in `OUT`. Print its size in kB (1 kB = 1,000 bytes), then the first 300 characters of its text.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides_path = OUT / "b3_landslides.geojson"
slides_gdf.to_file(slides_path)
print(slides_path.stat().st_size / 1000)
print(slides_path.read_text()[:300])
```

About 9.5 kB. The text is readable: each feature has its `properties` (the attributes) and a `geometry` with `"coordinates": [86.1506, 27.872]`, longitude first. That readability is GeoJSON's great strength.
</details>

## 7. Reading files back, and layers

Reading a file back is a good habit (Lesson A9): it proves the file holds what you think it holds.

A **layer** is one table of features of one geometry type, such as the districts or the landslides. A GeoJSON or Shapefile holds one layer. A GeoPackage can hold many, which makes it a tidy way to keep a whole project in one file:

- `gdf.to_file(path, layer="name")` adds or replaces one layer,
- `gpd.list_layers(path)` lists the layers in the file,
- `gpd.read_file(path, layer="name")` reads one of them.

`df.rename(columns={"old": "new"})` returns a table with renamed columns. It repairs names that a Shapefile cut short.

A plain CSV has no geometry type and no CRS. If you share points as CSV, keep the `lon` and `lat` columns and say in the file name or a note that they are EPSG:4326.

In [ ]:
back_shp = gpd.read_file(OUT / "b3_districts.shp")
back_gpkg = gpd.read_file(OUT / "b3_nepal.gpkg", layer="districts")
print(back_shp.columns.tolist())
print(back_gpkg.columns.tolist())
print(len(back_shp), len(back_gpkg), back_shp.crs)

In [ ]:
slides_gdf.to_file(OUT / "b3_nepal.gpkg", layer="landslides")
gpd.list_layers(OUT / "b3_nepal.gpkg")

Both files hold all 77 districts in EPSG:4326, but the Shapefile came back with the short names `district_c` and `province_c`. Code that asks for `district_code` would now fail with a `KeyError`. The GeoPackage kept the names whole.

The GeoPackage now holds two layers: `districts` (MultiPolygon) and `landslides` (Point).

### Practice

**Q1 · one line:** Read the `landslides` layer back from `b3_nepal.gpkg` and print how many rows it has.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
len(gpd.read_file(OUT / "b3_nepal.gpkg", layer="landslides"))
```

40 rows, the same as the CSV file: nothing was lost on the way.
</details>

**Q2 · two lines:** Use `rename` to give `back_shp` its full column names back, store the result in `fixed`, and print its columns.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fixed = back_shp.rename(columns={"district_c": "district_code", "province_c": "province_code"})
print(fixed.columns.tolist())
```

The columns are `district`, `district_code`, `province_code` and `geometry` again. You had to know the full names; a Shapefile alone cannot tell you.
</details>

**Q3 · three lines:** Add `peaks_gdf` to the GeoPackage as a third layer called `peaks`, then list the layers.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks_gdf.to_file(OUT / "b3_nepal.gpkg", layer="peaks")
gpd.list_layers(OUT / "b3_nepal.gpkg")
```

The file now lists three layers: districts, landslides and peaks. One file, one project, and no sidecar files to lose.
</details>

### Try this at home when you are online

Lessons in this course never need the internet. But a huge amount of free map data lives in **OpenStreetMap** (OSM), built by volunteers. The `osmnx` library downloads it straight into a GeoDataFrame. When you are online, try this in a new notebook:

```python
import osmnx as ox

# every building mapped in OpenStreetMap within Pokhara (this can take a minute)
buildings = ox.features_from_place("Pokhara, Nepal", tags={"building": True})
print(len(buildings))
buildings.plot(figsize=(8, 8))
```

The result is an ordinary GeoDataFrame: everything from this lesson works on it. OSM data is shared under the Open Database License, so credit "© OpenStreetMap contributors" on any map you publish.

## 8. When it breaks: a file that is not there

One letter is missing from the file name in the cell below. Run it.

In [ ]:
gpd.read_file(DATA / "nepal_district.geojson")

The traceback is long, because the error comes from deep inside the library that reads files. Go to the last line:

`DataSourceError: ../data/nepal_district.geojson: No such file or directory`

- `DataSourceError` is the reading library's name for "I could not open this as a data source".
- `No such file or directory` is the reason: it is a missing file, like `FileNotFoundError` in Lesson A7.

Check the name against the real files: `for p in DATA.glob("*.geojson"): print(p.name)` lists them (Lesson A7). Here the file is `nepal_districts.geojson`, with an **s**.

The same error appears when a Shapefile's `.shp` is missing, or when you are in a different working folder than you think (`Path.cwd()`, Lesson A7).

**Try it:** fix the file name.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · From CSV to GeoPackage and back

Read `practice_landslides.csv`, turn it into a GeoDataFrame in EPSG:4326, and save it as the layer `landslides` in a new file `b3_review.gpkg` in `OUT`. Read it back and print: the number of rows, the CRS, and whether the original and the copy have the same columns.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
records = pd.read_csv(DATA / "practice_landslides.csv")
points = gpd.points_from_xy(records["lon"], records["lat"])
layer = gpd.GeoDataFrame(records, geometry=points, crs="EPSG:4326")

path = OUT / "b3_review.gpkg"
layer.to_file(path, layer="landslides")
copy = gpd.read_file(path, layer="landslides")

print(len(copy), copy.crs)
print(layer.columns.tolist() == copy.columns.tolist())
```

40 rows in EPSG:4326, and `True`: the GeoPackage kept every column and its name. Comparing two lists with `==` checks them item by item.
</details>

### Review 2 · One province in three formats

Select the districts of province NP03. Save them as `b3_np03.geojson`, `b3_np03.gpkg` and `b3_np03.shp` in `OUT`. Then list every file whose name starts with `b3_np03`, with its size in kB, and add up the sizes of the Shapefile's parts.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
np03 = districts[districts["province_code"] == "NP03"]
np03.to_file(OUT / "b3_np03.geojson")
np03.to_file(OUT / "b3_np03.gpkg", layer="districts")
np03.to_file(OUT / "b3_np03.shp")

shapefile_bytes = 0
for path in sorted(OUT.glob("b3_np03.*")):
    size_kb = path.stat().st_size / 1000
    print(f"{path.name:18} {size_kb:7.1f} kB")
    if path.suffix in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        shapefile_bytes += path.stat().st_size
print(f"Shapefile parts together: {shapefile_bytes / 1000:.1f} kB")
```

Seven files appear: one GeoJSON, one GeoPackage and the five parts of the Shapefile, 51.6 kB together. Surprise: for this small layer the GeoPackage (155.6 kB) is larger than the GeoJSON (119.6 kB). A GeoPackage is a small database, and its fixed structure only pays off for larger layers. `path.suffix` (Lesson A7) picks out the Shapefile's parts. The same truncation warnings appear, because the columns are the same.
</details>

### Review 3 · A map of landslides by trigger

Draw the districts in light grey, then the landslides from `slides_gdf` coloured by `trigger` with `column="trigger"`, `categorical=True` and a legend. Zoom to the landslides with `ax.set_xlim` and `ax.set_ylim`, using their `total_bounds` with a margin of 0.3°. Add a title that says the data are practice data.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
min_x, min_y, max_x, max_y = slides_gdf.total_bounds

fig, ax = plt.subplots(figsize=(10, 5))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
slides_gdf.plot(ax=ax, column="trigger", categorical=True, legend=True,
                cmap="Set1", markersize=25)
ax.set_xlim(min_x - 0.3, max_x + 0.3)
ax.set_ylim(min_y - 0.3, max_y + 0.3)
ax.set_title("Landslides by trigger (practice data)")
plt.show()
```

`total_bounds` has four numbers, so they unpack into four names. The map zooms from about 83.1° to 88.2° E, where all 40 points are, and the district borders give them context. `Set1` has strong colours, good for a few points.
</details>

### Review 4 · 77 districts, 75 headquarters

Read `district_headquarters.geojson` into `hq_points`. Print the number of districts and of headquarters. Then draw all districts in light grey, and the headquarters as small black points. Show in orange the four districts made by the recent splits: Nawalparasi East, Nawalparasi West, Rukum East and Rukum West.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq_points = gpd.read_file(DATA / "district_headquarters.geojson")
print(len(districts), len(hq_points))

names = districts["district"]
split = (names == "Nawalparasi East") | (names == "Nawalparasi West") | (names == "Rukum East") | (names == "Rukum West")

fig, ax = plt.subplots(figsize=(10, 6))
districts.plot(ax=ax, color="lightgrey", edgecolor="white")
districts[split].plot(ax=ax, color="tab:orange", edgecolor="white")
hq_points.plot(ax=ax, color="black", markersize=8)
ax.set_title("District headquarters, and the four districts made by splits")
plt.show()
```

77 districts but 75 headquarters. The orange districts show where the two layers disagree: each orange pair has only one headquarters point, because the headquarters layer is older than the split. Data that is out of date is a common, quiet source of error, so always check when a layer was made.
</details>

## Recap

- A **GeoDataFrame** is a pandas table with a **geometry column**; every Lesson A9 tool works on it, and filtered rows keep their shapes.
- `gpd.read_file(path)` reads GeoJSON, GeoPackage or Shapefile. Inspect `.crs`, `.geom_type` and `.total_bounds` before you analyse.
- `gpd.points_from_xy(df["lon"], df["lat"])` turns a CSV into points; state the CRS yourself, because a CSV carries none.
- `.plot(column=..., categorical=True, legend=True)` colours by an attribute; `ax=ax` stacks layers, last on top.
- `to_file` writes **GeoJSON** (one readable text file), **Shapefile** (several files, 10-character column names) or **GeoPackage** (one file, many **layers**).

### Words you learned

| Word | Meaning |
| --- | --- |
| GeoPandas | The Python library for tables of geographic features; loaded as `gpd` |
| GeoDataFrame | A pandas DataFrame with a geometry column |
| geometry column | The column of a GeoDataFrame that holds the shapes |
| `read_file` | Reads a vector file into a GeoDataFrame |
| `total_bounds` | `[min_x, min_y, max_x, max_y]` of a whole layer |
| MultiPolygon | One feature made of one or more polygon parts |
| colour map (`cmap`) | A named set of colours used to colour a map, such as `Set2` |
| `points_from_xy` | Turns x (longitude) and y (latitude) columns into point geometries |
| GeoJSON | A text file format for features, readable by people and web maps |
| Shapefile | An old, widely used format made of several files (.shp, .shx, .dbf, .prj, .cpg) |
| GeoPackage | A one-file format, a small database, that can hold many layers |
| layer | One table of features of one geometry type, such as districts or landslides |
| `.prj` file | The part of a Shapefile that stores the CRS as text |
| warning | A message that something may be wrong, while the code still runs |
| `rename` | Returns a table with some columns renamed |
| OpenStreetMap | A free world map built by volunteers, downloadable with `osmnx` |

## Next

**B4 · Coordinate reference systems in practice:** what `EPSG:4326` really means and how to convert your layers to metres. Then areas, lengths and buffers come out in km instead of degrees.

<!-- pygile-nav -->
**PyGILE Course** · lesson 13 of 21 · [Course home](../README.md)  
← Previous: [B2 · Points, lines and polygons: geometry with shapely](B02_Points_lines_and_polygons.ipynb) · Next: [B4 · Coordinate reference systems in practice: measuring Nepal in metres](B04_Coordinate_reference_systems.ipynb) →